# Disentangling in-language signal from data volume: ablation, per-label, and multi-seed analysis

Follow-up experiments that isolate the source of the augmentation gain and
quantify its reliability. The matched-data arm in the main study (English plus
machine-translated target NLI) improves transfer, but mixing English and target
data leaves open whether the gain comes from the in-language signal or simply
from training volume relative to the baseline. Two volume-matched ablation arms
settle this:

- **english_only_40k** -- 40k English examples (same volume as the source arms,
  no in-language data),
- **target_only_20k** -- 20k machine-translated target examples (in-language,
  no English mixing),
- **matched_20k_20k** -- 20k English + 20k target (the main-study augmentation arm),
- **baseline_en** -- the standard 40k English baseline.

Comparing these attributes the gain to in-language signal vs. volume. Each
configuration is run over three seeds for confidence intervals, and every
evaluation records a full confusion matrix so per-label accuracy is available for
all targets. Set `QUICK_SMOKE = True` to validate the pipeline before the full
run.

## Setup

In [ ]:
!pip -q install "transformers>=4.44" "datasets>=2.20" "accelerate>=0.33" \
                "sentencepiece" "scikit-learn" "lang2vec" 2>/dev/null

In [ ]:
import os, time, json, warnings, random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)
from sklearn.metrics import accuracy_score, confusion_matrix
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
USE_FP16 = DEVICE == "cuda" and not USE_BF16
print("device:", DEVICE, "| precision:", "bf16" if USE_BF16 else "fp16" if USE_FP16 else "fp32")

## Configuration

In [ ]:
QUICK_SMOKE = False
XNLI_REPO   = "facebook/xnli"
OUTDIR      = "outputs_ablation"
os.makedirs(f"{OUTDIR}/figs", exist_ok=True)

MODELS  = {"xlmr": "xlm-roberta-base", "qwen": "Qwen/Qwen2.5-0.5B"}
TARGETS = ["ar", "el", "hi", "sw", "th", "ur"]
SEEDS   = [42, 1, 2]

# Four training configurations. Volume is held at 40k except target_only (20k),
# which is the in-language-but-smaller condition; comparing it to baseline_en
# (40k) and matched (40k total) separates signal from volume.
ARMS = {
    "baseline_en":     {"en": 40000},
    "matched_20k_20k": {"en": 20000, "TARGET": 20000},
    "english_only_40k":{"en": 40000},
    "target_only_20k": {"TARGET": 20000},
}

MAX_LEN, NUM_EPOCHS, LR = 128, 2, 2e-5
EVAL_SAMPLE_SIZE = 2000
BATCH = {"xlmr": 32, "qwen": 16}
LABELS = ["entailment", "neutral", "contradiction"]

if QUICK_SMOKE:
    MODELS  = {"xlmr": "xlm-roberta-base"}
    TARGETS = ["ur", "sw"]
    SEEDS   = [42]
    ARMS = {k: {kk: 400 for kk in v} for k, v in ARMS.items()}
    EVAL_SAMPLE_SIZE, NUM_EPOCHS = 300, 1

# baseline_en and english_only_40k are identical training sets; train once, reuse.
import datasets as _ds
_orig = getattr(_ds, "_orig_load_dataset", _ds.load_dataset); _ds._orig_load_dataset = _orig
def load_dataset(path, *a, **k):
    if path == "xnli": path = XNLI_REPO
    return _orig(path, *a, **k)
_ds.load_dataset = load_dataset
print("targets:", TARGETS, "| seeds:", SEEDS, "| arms:", list(ARMS))

## Helpers

In [ ]:
def load_nli(lang, split, n, seed):
    ds = load_dataset(XNLI_REPO, lang, split=split)
    return ds.shuffle(seed=seed).select(range(n)) if n and n < len(ds) else ds

def make_tokenizer(name):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    return tok

def tokenize_ds(ds, tok):
    def f(b): return tok(b["premise"], b["hypothesis"], truncation=True, max_length=MAX_LEN)
    return ds.map(f, batched=True, remove_columns=[c for c in ds.column_names if c != "label"])

def build_model(name):
    m = AutoModelForSequenceClassification.from_pretrained(name, num_labels=3)
    if m.config.pad_token_id is None:
        m.config.pad_token_id = make_tokenizer(name).pad_token_id
    return m

def build_train_set(spec, target, seed):
    parts = []
    for key, n in spec.items():
        lang = target if key == "TARGET" else key
        parts.append(load_nli(lang, "train", n, seed))
    return concatenate_datasets(parts).shuffle(seed=seed) if len(parts) > 1 else parts[0]

def fine_tune(name, train_ds, tag, seed):
    set_seed(seed)
    tok = make_tokenizer(name); model = build_model(name)
    args = TrainingArguments(
        output_dir=f"{OUTDIR}/ckpt_{tag}",
        per_device_train_batch_size=BATCH.get(tag.split("_")[0], 16),
        gradient_accumulation_steps=2, num_train_epochs=NUM_EPOCHS, learning_rate=LR,
        fp16=USE_FP16, bf16=USE_BF16, logging_steps=100, save_strategy="no",
        report_to="none", seed=seed)
    tr = Trainer(model=model, args=args, train_dataset=tokenize_ds(train_ds, tok),
                 data_collator=DataCollatorWithPadding(tok))
    tr.train()
    return tr, tok

def evaluate_on(trainer, tok, lang, seed):
    test = load_nli(lang, "test", EVAL_SAMPLE_SIZE, seed)
    pred = trainer.predict(tokenize_ds(test, tok))
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    y, yhat = np.array(test["label"]), logits.argmax(-1)
    cm = confusion_matrix(y, yhat, labels=[0, 1, 2])
    return accuracy_score(y, yhat) * 100, cm

def free():
    if DEVICE == "cuda": torch.cuda.empty_cache()

## Run all arms over all targets and seeds

Records accuracy and the confusion matrix for every (model, target, arm, seed).
`baseline_en` and `english_only_40k` share a training set, so the model is
trained once per (model, seed) and evaluated for both, halving the English-only
training cost.

In [ ]:
acc_rows, cm_rows = [], []

def record(model, target, arm, seed, acc, cm):
    acc_rows.append({"model": model, "target": target, "arm": arm,
                     "seed": seed, "acc": round(acc, 2)})
    cm_rows.append({"model": model, "target": target, "arm": arm, "seed": seed,
                    "cm": json.dumps(cm.tolist())})

for tag, mname in MODELS.items():
    for seed in SEEDS:
        # English-only model serves both baseline_en and english_only_40k
        print(f"\n[{tag} seed={seed}] training English-40k")
        tr_en, tok_en = fine_tune(mname, build_train_set({"en": 40000}, None, seed),
                                  f"{tag}_en40_{seed}", seed)
        for t in TARGETS:
            acc, cm = evaluate_on(tr_en, tok_en, t, seed)
            record(tag, t, "baseline_en", seed, acc, cm)
            record(tag, t, "english_only_40k", seed, acc, cm)
        del tr_en; free()

        # target-specific arms: matched and target-only
        for t in TARGETS:
            for arm in ["matched_20k_20k", "target_only_20k"]:
                print(f"[{tag} seed={seed}] {arm}: {t}")
                tr, tok = fine_tune(mname, build_train_set(ARMS[arm], t, seed),
                                    f"{tag}_{arm}_{t}_{seed}", seed)
                acc, cm = evaluate_on(tr, tok, t, seed)
                record(tag, t, arm, seed, acc, cm)
                print(f"   acc={acc:.2f}")
                del tr; free()

acc_df = pd.DataFrame(acc_rows); acc_df.to_csv(f"{OUTDIR}/ablation_acc.csv", index=False)
cm_df = pd.DataFrame(cm_rows);  cm_df.to_csv(f"{OUTDIR}/ablation_confusion.csv", index=False)
acc_df.head()


[xlmr seed=42] training English-40k


README.md:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/50.2M [00:00<?, ?B/s]

en/test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

en/validation-00000-of-00001.parquet:   0%|          | 0.00/157k [00:00<?, ?B/s]

...

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Step,Training Loss
100,2.787747
200,2.284631
300,2.185796
400,2.181024
500,2.125167
600,2.100974
700,2.087300
800,2.064241
900,2.035489
1000,2.039722


   acc=51.00


,model,target,arm,seed,acc
0,xlmr,ar,baseline_en,42,65.2
1,xlmr,ar,english_only_40k,42,65.2
2,xlmr,el,baseline_en,42,69.0
3,xlmr,el,english_only_40k,42,69.0
4,xlmr,hi,baseline_en,42,64.8


## Ablation analysis: signal vs. volume

The key contrasts, each averaged over targets and seeds:

- **matched** vs **baseline_en**: the augmentation gain reported in the main study.
- **english_only_40k** vs **baseline_en**: should be $\approx 0$ (same data); a
  sanity check that volume alone, in English, does nothing.
- **target_only_20k** vs **baseline_en**: in-language data at *half* the volume.
  If this still improves over the 40k English baseline, the gain is driven by
  in-language signal, not volume.
- **matched** vs **target_only_20k**: whether mixing English on top of
  in-language data adds anything.

In [ ]:
acc_df = pd.read_csv(f"{OUTDIR}/ablation_acc.csv")
# mean over seeds per (model, target, arm)
cell = acc_df.groupby(["model","target","arm"]).acc.mean().reset_index()
wide = cell.pivot_table(index=["model","target"], columns="arm", values="acc").reset_index()

def ci95(vals):
    vals = np.asarray(vals); m = vals.mean()
    se = vals.std(ddof=1)/np.sqrt(len(vals)) if len(vals) > 1 else 0.0
    return m, 1.96*se

from scipy import stats
print("=== per-model arm means (avg over targets+seeds) with 95% CI over seeds ===")
for tag in acc_df.model.unique():
    print(f"\n{tag}:")
    for arm in ["baseline_en","english_only_40k","target_only_20k","matched_20k_20k"]:
        # seed-level means (avg over targets within seed), then CI across seeds
        per_seed = (acc_df[(acc_df.model==tag)&(acc_df.arm==arm)]
                    .groupby("seed").acc.mean().values)
        m, ci = ci95(per_seed)
        print(f"  {arm:18s}: {m:5.2f}  +/- {ci:4.2f}")

print("\n=== paired contrasts (over model x target cells, seed-averaged) ===")
def contrast(a, b, label):
    A = wide[a].values; B = wide[b].values
    d = A - B; t, p = stats.ttest_rel(A, B)
    print(f"{label:42s}: mean {d.mean():+.2f}, wins {int((d>0).sum())}/{len(d)}, "
          f"t={t:.2f}, p={p:.4f}")
contrast("matched_20k_20k","baseline_en",      "matched vs baseline (augmentation gain)")
contrast("english_only_40k","baseline_en",     "english-40k vs baseline (volume sanity, ~0)")
contrast("target_only_20k","baseline_en",      "target-20k vs baseline (in-language @ half volume)")
contrast("matched_20k_20k","target_only_20k",  "matched vs target-only (does English help?)")
wide.round(2)

## Per-label accuracy across all targets

Aggregates the confusion matrices (summed over seeds) for each target under the
baseline and the matched-data arm, and reports per-label recall. This tests
whether augmentation specifically recovers entailment, the label the error
analysis flagged as the baseline's weak point.

In [ ]:
cm_df = pd.read_csv(f"{OUTDIR}/ablation_confusion.csv")
def agg_cm(model, target, arm):
    sub = cm_df[(cm_df.model==model)&(cm_df.target==target)&(cm_df.arm==arm)]
    total = np.zeros((3,3), dtype=int)
    for s in sub.cm: total += np.array(json.loads(s))
    return total

def per_label_recall(cm):
    return [cm[i,i]/cm[i].sum()*100 if cm[i].sum() else float("nan") for i in range(3)]

rows = []
for tag in cm_df.model.unique():
    for t in TARGETS:
        for arm in ["baseline_en","matched_20k_20k"]:
            cm = agg_cm(tag, t, arm)
            r = per_label_recall(cm)
            rows.append({"model":tag,"target":t,"arm":arm,
                         "entail":round(r[0],1),"neutral":round(r[1],1),
                         "contra":round(r[2],1)})
pl = pd.DataFrame(rows); pl.to_csv(f"{OUTDIR}/per_label.csv", index=False)

# entailment recovery: matched - baseline, per model x target
print("=== entailment recall: baseline -> matched (per target) ===")
for tag in cm_df.model.unique():
    print(f"\n{tag}:")
    for t in TARGETS:
        b = pl[(pl.model==tag)&(pl.target==t)&(pl.arm=="baseline_en")].entail.values[0]
        m = pl[(pl.model==tag)&(pl.target==t)&(pl.arm=="matched_20k_20k")].entail.values[0]
        print(f"  {t}: {b:5.1f} -> {m:5.1f}  ({m-b:+.1f})")
pl

## Figures

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family":"serif","font.size":9,"savefig.dpi":300,"axes.linewidth":0.8})
acc_df = pd.read_csv(f"{OUTDIR}/ablation_acc.csv")

# Ablation bar chart with seed CIs, per model
arms = ["baseline_en","english_only_40k","target_only_20k","matched_20k_20k"]
arm_lbl = ["baseline\n(40k en)","en-only\n40k","target\n20k","matched\n20+20k"]
cols = ["#264653","#8d99ae","#e76f51","#2a9d8f"]
models = list(acc_df.model.unique())
fig, axes = plt.subplots(1, len(models), figsize=(2.7*len(models), 3))
if len(models) == 1: axes = [axes]
for ax, tag in zip(axes, models):
    means, cis = [], []
    for arm in arms:
        per_seed = (acc_df[(acc_df.model==tag)&(acc_df.arm==arm)]
                    .groupby("seed").acc.mean().values)
        m = per_seed.mean()
        se = per_seed.std(ddof=1)/np.sqrt(len(per_seed)) if len(per_seed)>1 else 0
        means.append(m); cis.append(1.96*se)
    ax.bar(range(4), means, yerr=cis, capsize=3, color=cols, edgecolor="k", linewidth=.4)
    ax.axhline(means[0], color="0.5", lw=.7, ls="--")
    ax.set_xticks(range(4)); ax.set_xticklabels(arm_lbl, fontsize=6.5)
    ax.set_title(tag, fontsize=9)
    lo = min(means)-max(cis)-2
    ax.set_ylim(max(0, lo), None)
    for s in ["top","right"]: ax.spines[s].set_visible(False)
axes[0].set_ylabel("zero-shot accuracy (%)")
plt.tight_layout(); plt.savefig(f"{OUTDIR}/figs/fig_ablation.png", bbox_inches="tight"); plt.show()